# Capstone Step 2: Preprocessing & Feature Engineering for Deep Learning

**Module 2 : Stock Market Deep Learning System**

Objective: transform raw OHLCV data into model-ready features, with proper temporal train/validation/test splits, technical indicators, and normalization.

This notebook picks up directly from `01_eda.ipynb`, using the same data files.

In [1]:
import os
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
import joblib

import ta  # pure-Python technical analysis library

DATA_DIR = "../data"
PROCESSED_DIR = "../data/processed"
os.makedirs(PROCESSED_DIR, exist_ok=True)

pd.set_option("display.max_columns", None)
torch.manual_seed(42)
np.random.seed(42)

## 1. Load Data

Reload the raw stock and VIX data. We recompute log returns here (rather than importing
from `01_eda.ipynb`) so this notebook is fully reproducible on its own.

In [2]:
stocks = pd.read_csv(os.path.join(DATA_DIR, "sp500_stocks.csv"), parse_dates=["Date"])
vix = pd.read_csv(os.path.join(DATA_DIR, "vix.csv"), parse_dates=["Date"])

price_col = "Adj Close" if "Adj Close" in stocks.columns else "Close"
stocks = stocks.sort_values(["Ticker", "Date"]).reset_index(drop=True)
stocks["LogReturn"] = stocks.groupby("Ticker")[price_col].transform(
    lambda s: np.log(s / s.shift(1))
)

vix_col = "Close" if "Close" in vix.columns else vix.columns[1]
vix = vix[["Date", vix_col]].rename(columns={vix_col: "VIX"}).sort_values("Date")

print("Stocks shape:", stocks.shape)
print("VIX shape:", vix.shape)
stocks.head()

Stocks shape: (259073, 8)
VIX shape: (5282, 2)


,Date,Close,High,Low,Open,Volume,Ticker,LogReturn
0,2005-01-03,0.946491,0.973709,0.936172,0.968773,691992000,AAPL,NaN
1,2005-01-04,0.956211,0.979091,0.941704,0.953967,1096810400,AAPL,0.010217
2,2005-01-05,0.964586,0.975802,0.957856,0.963987,680433600,AAPL,0.008720
3,2005-01-06,0.965334,0.970717,0.947089,0.967128,705555200,AAPL,0.000775
4,2005-01-07,1.035621,1.041304,0.968324,0.972063,2227450400,AAPL,0.070282


## 2. Technical Indicators

Compute RSI(14), MACD(12,26,9), Bollinger Band width(20), ATR(14), OBV, and a
5-day/20-day moving-average crossover signal, per ticker. All indicators are computed
using only past and current data at each point in time (the `ta` library computes
indicators causally by construction, no future leakage).

We also merge in VIX level and VIX daily change as regime features, motivated directly
by the Step 1 EDA finding that VIX has real (if imperfect) predictive value for future
volatility.

In [3]:
def compute_indicators_for_group(g):
    """Compute technical indicators for one ticker's rows. Returns only the new
    columns, indexed identically to the input, so they can be concatenated back onto
    the original DataFrame without depending on pandas' groupby-apply column-retention
    behavior (which changed across pandas versions)."""
    close = g[price_col]
    high = g["High"]
    low = g["Low"]
    volume = g["Volume"]

    out = pd.DataFrame(index=g.index)

    # RSI(14)
    out["RSI_14"] = ta.momentum.RSIIndicator(close=close, window=14).rsi()

    # MACD(12, 26, 9)
    macd = ta.trend.MACD(close=close, window_slow=26, window_fast=12, window_sign=9)
    out["MACD"] = macd.macd()
    out["MACD_signal"] = macd.macd_signal()
    out["MACD_hist"] = macd.macd_diff()

    # Bollinger Band width (20)
    bb = ta.volatility.BollingerBands(close=close, window=20, window_dev=2)
    out["BB_width"] = (bb.bollinger_hband() - bb.bollinger_lband()) / bb.bollinger_mavg()

    # ATR(14)
    out["ATR_14"] = ta.volatility.AverageTrueRange(
        high=high, low=low, close=close, window=14
    ).average_true_range()

    # OBV
    out["OBV"] = ta.volume.OnBalanceVolumeIndicator(close=close, volume=volume).on_balance_volume()

    # Moving average crossover: 5-day SMA minus 20-day SMA
    sma5 = close.rolling(5, min_periods=5).mean()
    sma20 = close.rolling(20, min_periods=20).mean()
    out["SMA_crossover"] = sma5 - sma20

    # 200-day SMA, needed for the market breadth regime feature below
    sma200 = close.rolling(200, min_periods=200).mean()
    out["Above_SMA_200"] = (close > sma200).astype(float)
    out.loc[sma200.isna(), "Above_SMA_200"] = np.nan

    return out

# Sort once so each per-ticker group is in chronological order before computing
# rolling/indicator windows.
stocks = stocks.sort_values(["Ticker", "Date"]).reset_index(drop=True)
indicator_cols = stocks.groupby("Ticker", group_keys=False).apply(
    compute_indicators_for_group
)
stocks = pd.concat([stocks, indicator_cols], axis=1)

stocks = stocks.merge(vix, on="Date", how="left")
stocks["VIX_change"] = stocks.groupby("Ticker")["VIX"].transform(lambda s: s.diff())

feature_cols = [
    "LogReturn", "RSI_14", "MACD", "MACD_signal", "MACD_hist",
    "BB_width", "ATR_14", "OBV", "SMA_crossover", "VIX", "VIX_change",
    "MarketBreadth_200", "Volume",
]
stocks[["Date", "Ticker"] + [c for c in feature_cols if c in stocks.columns]].head(25)

,Date,Ticker,LogReturn,RSI_14,MACD,MACD_signal,MACD_hist,BB_width,ATR_14,OBV,SMA_crossover,VIX,VIX_change,Volume
0,2005-01-03,AAPL,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,691992000,NaN,14.08,NaN,691992000
1,2005-01-04,AAPL,0.010217,NaN,NaN,NaN,NaN,NaN,0.000000,1788802400,NaN,13.98,-0.100000,1096810400
2,2005-01-05,AAPL,0.008720,NaN,NaN,NaN,NaN,NaN,0.000000,2469236000,NaN,14.09,0.110001,680433600
3,2005-01-06,AAPL,0.000775,NaN,NaN,NaN,NaN,NaN,0.000000,3174791200,NaN,13.58,-0.510000,705555200
4,2005-01-07,AAPL,0.070282,NaN,NaN,NaN,NaN,NaN,0.000000,5402241600,NaN,13.49,-0.090000,2227450400
5,2005-01-10,AAPL,-0.004196,NaN,NaN,NaN,NaN,NaN,0.000000,3676932000,NaN,13.23,-0.260000,1725309600
6,2005-01-11,AAPL,-0.065932,NaN,NaN,NaN,NaN,NaN,0.000000,1065304800,NaN,13.19,-0.040000,2611627200
7,2005-01-12,AAPL,0.013845,NaN,NaN,NaN,NaN,NaN,0.000000,2985007200,NaN,12.56,-0.629999,1919702400
8,2005-01-13,AAPL,0.064194,NaN,NaN,NaN,NaN,NaN,0.000000,6149724000,NaN,12.84,0.280000,3164716800
9,2005-01-14,AAPL,0.005714,NaN,NaN,NaN,NaN,NaN,0.000000,7920466400,NaN,12.43,-0.410000,1770742400


**Note on OBV and Volume scale:** both are raw share counts and can span many orders of magnitude across tickers. We log-transform Volume before normalization (Section 5) so the network trains on an approximately symmetric distribution, per the module's guidance to log-transform volume and dollar-denominated features.

### 2.1 Market Breadth (Regime Feature)

Market breadth is the percentage of our 50 tickers trading above their own 200-day SMA
on a given day. It is a market-wide regime signal, not ticker-specific, computed
cross-sectionally at each date using only that date's per-ticker `Above_SMA_200` flags
(each of which is itself causal, per-ticker, as computed above). Every ticker sees the
same breadth value for a given date, since breadth describes the overall market
environment.

In [4]:
breadth = (
    stocks.dropna(subset=["Above_SMA_200"])
    .groupby("Date")["Above_SMA_200"]
    .mean()
    .rename("MarketBreadth_200")
    .reset_index()
)

stocks = stocks.merge(breadth, on="Date", how="left")

print("Market breadth sample (fraction of tickers above their 200-day SMA):")
stocks[["Date", "MarketBreadth_200"]].drop_duplicates().dropna().tail(10)

Market breadth sample (fraction of tickers above their 200-day SMA):


,Date,MarketBreadth_200
5272,2025-12-16,0.70
5273,2025-12-17,0.74
5274,2025-12-18,0.74
5275,2025-12-19,0.74
5276,2025-12-22,0.74
5277,2025-12-23,0.74
5278,2025-12-24,0.74
5279,2025-12-26,0.74
5280,2025-12-29,0.74
5281,2025-12-30,0.74


## 3. Target Variables

Two targets, computed per ticker:

- **Regression target:** `future_log_return`, the *next-day* log return (i.e. the target for day *t* is the log return realized on day *t+1*). This is a shift, not a recomputation, so no future information leaks into the feature set itself.
- **Classification target:** `future_direction_5d`, 1 if the *cumulative* log return over the next 5 trading days is positive, 0 otherwise.

Both targets are deliberately shifted *forward* relative to the features: at row *t*, every feature column reflects information available up to and including day *t*, while both targets describe what happens *after* day *t*. This is the boundary that keeps Capstone Step 2 free of look-ahead bias.

In [5]:
def add_targets(df):
    df = df.copy()
    # Next-day log return (regression target)
    df["future_log_return"] = df.groupby("Ticker")["LogReturn"].shift(-1)

    # 5-day forward cumulative log return -> direction (classification target)
    fwd_5d_cum = df.groupby("Ticker")[price_col].transform(
        lambda s: np.log(s.shift(-5) / s)
    )
    df["future_direction_5d"] = (fwd_5d_cum > 0).astype(float)
    # Rows where the 5-day forward window doesn't exist yet (last 5 rows per ticker)
    # must also be NaN, not a default 0/1 label
    df.loc[fwd_5d_cum.isna(), "future_direction_5d"] = np.nan

    return df

stocks = add_targets(stocks)
stocks[["Date", "Ticker", price_col, "future_log_return", "future_direction_5d"]].tail(10)

,Date,Ticker,Close,future_log_return,future_direction_5d
259063,2025-12-16,XOM,112.424690,0.023527,1.0
259064,2025-12-17,XOM,115.101013,-0.007438,1.0
259065,2025-12-18,XOM,114.248108,0.001286,1.0
259066,2025-12-19,XOM,114.395164,0.012434,1.0
259067,2025-12-22,XOM,115.826447,0.010692,1.0
259068,2025-12-23,XOM,117.071465,-0.001676,NaN
259069,2025-12-24,XOM,116.875412,-0.000923,NaN
259070,2025-12-26,XOM,116.767570,0.011851,NaN
259071,2025-12-29,XOM,118.159645,0.003809,NaN
259072,2025-12-30,XOM,118.610596,NaN,NaN


## 4. Temporal Train / Validation / Test Split

**We split by date, identically across all tickers, never by shuffling rows.**

Financial time series are autocorrelated: today's return is not independent of yesterday's, and market regimes persist over months. A random row-level split would let the model train on data from *after* a point it's later validated or tested on for a different ticker on a nearby date, an indirect form of look-ahead bias. Splitting temporally, with train strictly before validation strictly before test, guarantees the
model is only ever evaluated on genuinely unseen future dates, exactly as it would be in production.

Following the module's split: **train = first 70%, validation = next 15%, test = last 15%**, by calendar date (not by row count per ticker, so the split boundary is the same date for every ticker).

In [7]:
all_dates = np.sort(stocks["Date"].unique())
n = len(all_dates)
train_end = all_dates[int(n * 0.70)]
val_end = all_dates[int(n * 0.85)]

print(f"Train:      {all_dates[0]} to {train_end}")
print(f"Validation: {train_end} to {val_end}")
print(f"Test:       {val_end} to {all_dates[-1]}")

train_df = stocks[stocks["Date"] <= train_end].copy()
val_df = stocks[(stocks["Date"] > train_end) & (stocks["Date"] <= val_end)].copy()
test_df = stocks[stocks["Date"] > val_end].copy()

print(f"\nTrain rows:      {len(train_df):,}")
print(f"Validation rows: {len(val_df):,}")
print(f"Test rows:       {len(test_df):,}")

Train:      2005-01-03T00:00:00.000000 to 2019-09-11T00:00:00.000000
Validation: 2019-09-11T00:00:00.000000 to 2022-11-01T00:00:00.000000
Test:       2022-11-01T00:00:00.000000 to 2025-12-30T00:00:00.000000

Train rows:      179,873
Validation rows: 39,600
Test rows:       39,600


**Note on the validation window:** the validation split (2019-09 to 2022-11) contains the COVID crash, one of the two most extreme volatility events in the dataset (recall the 90-100% annualized volatility spikes observed for AAPL in the Step 1 EDA). This means weaker validation performance should not automatically be read as a poor model: it may simply reflect the genuine difficulty of this period rather than a modeling failure. This caveat is worth carrying into `model_documentation.md`, so validation-period results are not over-interpreted without this context.

## 5. Handle Missing Values from Indicator Warm-up

Every indicator has a warm-up period: RSI(14) needs 14 prior days, the 20-day SMA needs
20, MACD's slowest component needs 26, and so on. Rows before a ticker's warm-up period
are NaN by construction, and the last 1-5 rows of each ticker have NaN targets (no future
data to compute a forward return from). Both are dropped, not imputed: fabricating a
value here would either encode a fake pattern (if imputed with a constant) or leak
neighboring-day information (if interpolated).

In [8]:
required_cols = list(dict.fromkeys(feature_cols)) + ["future_log_return", "future_direction_5d"]

def drop_incomplete_rows(df, name):
    before = len(df)
    df = df.dropna(subset=required_cols).reset_index(drop=True)
    after = len(df)
    print(f"{name}: dropped {before - after:,} rows ({(before - after) / before * 100:.1f}%), "
          f"{after:,} remain")
    return df

train_df = drop_incomplete_rows(train_df, "Train")
val_df = drop_incomplete_rows(val_df, "Validation")
test_df = drop_incomplete_rows(test_df, "Test")

Train: dropped 9,452 rows (5.3%), 170,421 remain
Validation: dropped 0 rows (0.0%), 39,600 remain
Test: dropped 250 rows (0.6%), 39,350 remain


## 6. Feature Normalization

**The scaler is fit on training data only**, then applied unchanged to validation and test. Fitting on the full dataset (including future validation/test dates) would leak information about the future distribution of prices and volatility into training, a
subtle but common form of look-ahead bias.

Volume and OBV are log-transformed first (they span orders of magnitude and are strictly positive / can be negative for OBV, handled with a signed log), then all features are standardized (zero mean, unit variance) using `StandardScaler`.

In [9]:
def signed_log1p(x):
    """log1p that preserves sign, for OBV which can be negative."""
    return np.sign(x) * np.log1p(np.abs(x))

for df in (train_df, val_df, test_df):
    df["Volume_log"] = np.log1p(df["Volume"])
    df["OBV_log"] = signed_log1p(df["OBV"])

scaled_feature_cols = [
    "RSI_14", "MACD", "MACD_signal", "MACD_hist", "BB_width", "ATR_14",
    "OBV_log", "SMA_crossover", "VIX", "VIX_change", "MarketBreadth_200", "Volume_log",
]

scaler = StandardScaler()
scaler.fit(train_df[scaled_feature_cols])

for df in (train_df, val_df, test_df):
    df[scaled_feature_cols] = scaler.transform(df[scaled_feature_cols])

# Sanity check: train mean ~0, std ~1; val/test may differ (expected, different regime)
print("Train scaled features (mean, std):")
print(train_df[scaled_feature_cols].agg(["mean", "std"]).round(3))
print("\nValidation scaled features (mean, std) -- NOT necessarily 0/1, uses train's scaler:")
print(val_df[scaled_feature_cols].agg(["mean", "std"]).round(3))

joblib.dump(scaler, os.path.join(PROCESSED_DIR, "scaler.pkl"))
print("\nSaved scaler to", os.path.join(PROCESSED_DIR, "scaler.pkl"))

Train scaled features (mean, std):
      RSI_14  MACD  MACD_signal  MACD_hist  BB_width  ATR_14  OBV_log  \
mean    -0.0  -0.0          0.0        0.0      -0.0    -0.0     -0.0   
std      1.0   1.0          1.0        1.0       1.0     1.0      1.0   

      SMA_crossover  VIX  VIX_change  MarketBreadth_200  Volume_log  
mean           -0.0 -0.0         0.0               -0.0        -0.0  
std             1.0  1.0         1.0                1.0         1.0  

Validation scaled features (mean, std) -- NOT necessarily 0/1, uses train's scaler:
      RSI_14   MACD  MACD_signal  MACD_hist  BB_width  ATR_14  OBV_log  \
mean  -0.053  0.171        0.181      0.003     0.292   2.014    0.221   
std    1.026  3.179        3.162      3.308     1.144   2.644    0.853   

      SMA_crossover    VIX  VIX_change  MarketBreadth_200  Volume_log  
mean          0.108  0.586       0.008             -0.168      -0.113  
std           3.243  1.022       1.400              1.114       0.880  

Saved scal

**Sanity check interpretation:** the training set's scaled features should have mean ≈ 0 and std ≈ 1 by construction (that's what `StandardScaler.fit` does). The validation set will generally *not* be exactly 0/1, and that's expected and correct: it confirms we're using the training period's mean/std rather than silently re-fitting on validation data, which would defeat the purpose of a leakage-free split.

## 7. PyTorch Dataset & DataLoaders

Two datasets are provided: one for the regression target and one for the classification target, since they will train separate models later. Both draw from the samecross-sectional (ticker, day) rows, so shuffling is appropriate here, each row is already an independent (ticker, date) observation with no sequence structure. Sequence based datasets (sliding windows for the LSTM) are built separately in `04_sequence_models.ipynb`, where shuffling must be disabled.

In [11]:
class StockCrossSectionalDataset(Dataset):
    """Cross-sectional (ticker, day) dataset: each row is an independent sample."""

    def __init__(self, df, feature_cols, target_col):
        self.features = torch.tensor(df[feature_cols].values, dtype=torch.float32)
        self.targets = torch.tensor(df[target_col].values, dtype=torch.float32).unsqueeze(1)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return self.features[idx], self.targets[idx]


model_feature_cols = scaled_feature_cols + ["LogReturn"]  # include today's own return as a feature

train_reg_ds = StockCrossSectionalDataset(train_df, model_feature_cols, "future_log_return")
val_reg_ds = StockCrossSectionalDataset(val_df, model_feature_cols, "future_log_return")
test_reg_ds = StockCrossSectionalDataset(test_df, model_feature_cols, "future_log_return")

train_clf_ds = StockCrossSectionalDataset(train_df, model_feature_cols, "future_direction_5d")
val_clf_ds = StockCrossSectionalDataset(val_df, model_feature_cols, "future_direction_5d")
test_clf_ds = StockCrossSectionalDataset(test_df, model_feature_cols, "future_direction_5d")

BATCH_SIZE = 64

train_reg_loader = DataLoader(train_reg_ds, batch_size=BATCH_SIZE, shuffle=True)
val_reg_loader = DataLoader(val_reg_ds, batch_size=BATCH_SIZE, shuffle=False)
test_reg_loader = DataLoader(test_reg_ds, batch_size=BATCH_SIZE, shuffle=False)

train_clf_loader = DataLoader(train_clf_ds, batch_size=BATCH_SIZE, shuffle=True)
val_clf_loader = DataLoader(val_clf_ds, batch_size=BATCH_SIZE, shuffle=False)
test_clf_loader = DataLoader(test_clf_ds, batch_size=BATCH_SIZE, shuffle=False)

In [12]:
# Sanity check: pull one batch and confirm shapes
xb, yb = next(iter(train_reg_loader))
print("Feature batch shape:", xb.shape)
print("Target batch shape:", yb.shape)
print("Feature columns:", model_feature_cols)

Feature batch shape: torch.Size([64, 13])
Target batch shape: torch.Size([64, 1])
Feature columns: ['RSI_14', 'MACD', 'MACD_signal', 'MACD_hist', 'BB_width', 'ATR_14', 'OBV_log', 'SMA_crossover', 'VIX', 'VIX_change', 'MarketBreadth_200', 'Volume_log', 'LogReturn']


## 8. Save Preprocessed Data

Save the split, feature-engineered DataFrames (for later notebooks that need the rawDataFrame, e.g. backtesting) and the fitted scaler. Tensors are reconstructed on demand from these CSVs in subsequent notebooks, rather than pickled directly, so the pipelinestays framework-agnostic and inspectable.

In [13]:
train_df.to_csv(os.path.join(PROCESSED_DIR, "train.csv"), index=False)
val_df.to_csv(os.path.join(PROCESSED_DIR, "val.csv"), index=False)
test_df.to_csv(os.path.join(PROCESSED_DIR, "test.csv"), index=False)

with open(os.path.join(PROCESSED_DIR, "feature_columns.txt"), "w") as f:
    f.write("\n".join(model_feature_cols))

print("Saved to", PROCESSED_DIR + ":")
for fname in os.listdir(PROCESSED_DIR):
    size_kb = os.path.getsize(os.path.join(PROCESSED_DIR, fname)) / 1024
    print(f"  {fname} ({size_kb:.1f} KB)")

Saved to ../data/processed:
  scaler.pkl (1.2 KB)
  val.csv (15140.8 KB)
  test.csv (15083.8 KB)
  feature_columns.txt (0.1 KB)
  train.csv (65827.6 KB)


## Summary

- Computed RSI(14), MACD(12,26,9), Bollinger Band width(20), ATR(14), OBV, and a 5/20-day SMA crossover per ticker, plus VIX level, VIX change, and market breadth (% of tickers above their 200-day SMA) as regime features.
- Two targets defined with an explicit forward shift: next-day log return (regression) and 5-day-forward direction (classification), both strictly using only future data on the target side, never the feature side.
- Split temporally at the calendar-date level (70% / 15% / 15%), identical cutoff dates across all tickers, no shuffling across the split boundary.
- Dropped indicator warm-up NaNs and target NaNs (no imputation).
- Normalized with `StandardScaler` fit on training data only, log-transforming Volume and OBV first.
- Built PyTorch `Dataset`/`DataLoader` pairs (batch size 64) for both targets, shuffled for training since each row is an independent cross-sectional sample.
- Saved processed splits, the fitted scaler, and the feature column list to `data/processed/` for reuse in later notebooks.

**Next step:** `03_feedforward_nn.ipynb`: build and train a feedforward network on `future_log_return`, with dropout, batch normalization, and a linear-regression baseline for comparison.